In [12]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

#створення та розділення даних
data = load_iris()
x_train, x_test, y_train, y_test = train_test_split(data.data, data.target, test_size= 0.3, random_state= 42)

#типи підрозділів, індекс для зручності
names_of_divisions = {0: 'снайпери', 1: 'піхота', 2: 'танки'} 

#створення тактичних профілів
tactical_profiles = {}

for divisions_type in [0, 1, 2]:
    divisions_coords = x_train[y_train == divisions_type] #координати конкретного підрозділу

#бойовий порядок(коваріація)
    battle_order = np.cov(divisions_coords, rowvar= False)
    inv_battle_order = np.linalg.inv(battle_order)

#розрахунок параметрів штабів та ймовірностей
    tactical_profiles[divisions_type] = {'divis_center': np.mean(divisions_coords, axis= 0), #центр дислокації
    'inv_cov': inv_battle_order, #обернений БП 
    'det_cov': np.linalg.det(battle_order), #територія займана підрозділом
    'prior': len(divisions_coords)/len(x_train)} #ймовірність

#ідентифікація цілей/бойових одиниць(дискрим функц)
def goal_identification(target_x):
    scores = []
    for divisions_type in [0, 1, 2]:
        p = tactical_profiles[divisions_type]
        diff = target_x - p['divis_center'] #відхилення від дислокації
        #QDA
        score = np.log(p['prior']) - 0.5 * np.log(p['det_cov']) - 0.5 * diff@ p['inv_cov'] @diff
        scores.append(score)
    return np.argmax(scores) #повертаємо підрозділ звідки ймовірно БО

#тестова вибірка
results = np.array([goal_identification(x) for x in x_test])

#порівнюємо дані зі штабом
model_sk = QuadraticDiscriminantAnalysis().fit(x_train, y_train)
sk_results = model_sk.predict(x_test)

#діагностика с-тем розпізнавання
print(f'{'Обʼєкт №:':<7} | {'Доповідь розвідки:':<18} | {'Прогноз:':<15} | {'Штаб:'}')
print('-' * 70)


#порівняння
for i in range(15):
    in_fact = names_of_divisions[y_test[i]] #фактичний тип підрозділу
    forecast = names_of_divisions[results[i]] #наш прогноз
    headquarters = names_of_divisions[sk_results[i]]

#статус обробки
    status = 'Completed' if forecast == in_fact  else '!!!Diagnostic error!!!'

    print(f'Обʼєкт #{i+1:<2} | {in_fact:<18} | {forecast:<15} | {headquarters:<10} | {status}')



#фінальний рапорт(звітування)
accuracy = np.mean(results == y_test) * 100
print('-' * 70)
print(f'РАПОРТ: Точність діагностичної системи розпізнавання обʼєктів становить {accuracy:.1f}%')

Обʼєкт №: | Доповідь розвідки: | Прогноз:        | Штаб:
----------------------------------------------------------------------
Обʼєкт #1  | піхота             | піхота          | піхота     | Completed
Обʼєкт #2  | снайпери           | снайпери        | снайпери   | Completed
Обʼєкт #3  | танки              | танки           | танки      | Completed
Обʼєкт #4  | піхота             | піхота          | піхота     | Completed
Обʼєкт #5  | піхота             | піхота          | піхота     | Completed
Обʼєкт #6  | снайпери           | снайпери        | снайпери   | Completed
Обʼєкт #7  | піхота             | піхота          | піхота     | Completed
Обʼєкт #8  | танки              | танки           | танки      | Completed
Обʼєкт #9  | піхота             | піхота          | піхота     | Completed
Обʼєкт #10 | піхота             | піхота          | піхота     | Completed
Обʼєкт #11 | танки              | танки           | танки      | Completed
Обʼєкт #12 | снайпери           | снайпери     